# 01 — Structure mapping

Load UniProt↔PDB mapping hits for the example protein. Each hit is one PDB chain with one or more **fragments** (UniProt, PDB ATOM, SEQRES). Numbering is not a global offset: 7r0d chain AAA has a gap between residues 37 and 41.

Contacts over hits with ≥60% query identity are computed in notebook 02 (`select_structures`). The representative PDB in `config.json` is used later for SPURS only.

In [1]:
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.mapping import (
    fragment_table,
    load_structural_mapping_json,
    pdb_to_uniprot_map,
    read_fasta,
    select_mapping_entry,
    unique_pdb_ids,
)

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

header, uniprot_seq = read_fasta(EXAMPLE / f"{cfg['protein']}.fasta")
print(header, len(uniprot_seq), "aa")
print(uniprot_seq[:60], "...")

NUD15_HUMAN 164 aa
MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLPGGHLEFGETWEECA ...


## All homologous PDB/chain hits

In [2]:
entries = load_structural_mapping_json(EXAMPLE / "mapping" / "pdb_hits.json")
print("n chain hits:", len(entries))
print("unique PDB IDs:", unique_pdb_ids(entries))

frag_rows = []
for entry in entries:
    frags = fragment_table(entry)
    frags.insert(0, "pdb_id", entry["pdb_id"])
    frags.insert(1, "pdb_chain", entry["pdb_chain"])
    frag_rows.append(frags)
frag_all = pd.concat(frag_rows, ignore_index=True)
display(frag_all)
print("multi-fragment chains:")
display(frag_all.loc[frag_all.duplicated(["pdb_id", "pdb_chain"], keep=False)])

n chain hits: 30
unique PDB IDs: ['5bon', '5lpg', '6t5j', '7aom', '7aop', '7b63', '7b64', '7b65', '7b66', '7b7v', '7nr6', '7r0d']


,pdb_id,pdb_chain,fragment,uniprot_start,uniprot_end,pdb_start,pdb_end,n_residues,seqres_start,seqres_end
0,5bon,A,1,9,164,9,164,156,9,164
1,5bon,B,1,10,163,10,163,154,10,163
2,5bon,C,1,10,164,10,164,155,10,164
3,5bon,D,1,10,158,10,158,149,10,158
4,5bon,E,1,10,163,10,163,154,10,163
5,5bon,F,1,10,164,10,164,155,10,164
6,5bon,G,1,10,158,10,158,149,10,158
7,5bon,H,1,10,164,10,164,155,10,164
8,5lpg,A,1,9,164,9,164,156,9,164
9,5lpg,B,1,9,164,9,164,156,9,164


multi-fragment chains:


,pdb_id,pdb_chain,fragment,uniprot_start,uniprot_end,pdb_start,pdb_end,n_residues,seqres_start,seqres_end
28,7r0d,AAA,1,9,37,9,37,29,9,37
29,7r0d,AAA,2,41,163,41,163,123,41,163


## Representative entry (SPURS PDB)

In [3]:
entry = select_mapping_entry(
    entries,
    pdb_id=cfg["pdb_id"],
    chain=cfg["chain"],
    uniprot_id=cfg["uniprot_id"],
)
print("pdb / chain / UniProt:", entry["pdb_id"], entry["pdb_chain"], entry["uniprot_ac"], entry["uniprot_id"])
display(fragment_table(entry))
pdb2uni = pdb_to_uniprot_map(entry)
nums = sorted(pdb2uni)
print("PDB→UniProt map size:", len(pdb2uni))
print("first:", nums[0], "→", pdb2uni[nums[0]], " last:", nums[-1], "→", pdb2uni[nums[-1]])

pdb / chain / UniProt: 5lpg A Q9NV35 NUD15_HUMAN


,fragment,uniprot_start,uniprot_end,pdb_start,pdb_end,n_residues,seqres_start,seqres_end
0,1,9,164,9,164,156,9,164


PDB→UniProt map size: 156
first: 9 → 9  last: 164 → 164


To rebuild `pdb_hits.json` from SIFTS, download
[pdb_chain_uniprot.csv.gz](https://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/csv/pdb_chain_uniprot.csv.gz)
and call `mapping_entries_from_sifts` (one record per chain, one fragment per SIFTS row).